In [6]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

# Загружаем реальные данные
real_data = pd.read_csv('C:/Users/majkl/WorkSpace/Analisys_seller_mp_oz/anonymize_data/anonymous_seller_product.csv', sep=';', encoding='utf-8-sig')

# Создаем генератор данных
fake = Faker('ru_RU')
np.random.seed(42)

# Количество записей в итоговом датасете
n_records = 2354

# Выбираем случайные товары из реальных данных
selected_products = real_data.sample(n=n_records, replace=True).reset_index(drop=True)

# Создаем список уникальных номеров отправлений
order_numbers = [f"ORD{random.randint(100000, 999999)}" for _ in range(n_records)]

# Создаем датафрейм с привязкой к реальным данным
df = pd.DataFrame({
    'Номер отправления': order_numbers,
    'Статус отправления': np.random.choice(['Доставлен', 'В пути', 'Отменен'], n_records),
    'Ozon ID': selected_products['Ozon Product ID'].values,
    'Артикул товара': selected_products['Артикул'].values,
    'Итоговая стоимость': selected_products['Текущая цена с учетом скидки, ₽'].values,
    'Количество товаров в отправлении': np.random.randint(1, 5, n_records),
    'Скидка %': np.random.choice([0, 5, 10, 15, 20], n_records),
    'Акции': np.random.choice(['Черная пятница', 'Новогодняя распродажа', 'Нет'], n_records),
    'Регион доставки': [fake.city() for _ in range(n_records)],
    'Способ доставки': np.random.choice(['Курьер', 'ПВЗ', 'Постамат'], n_records)
})

# Рассчитываем сумму отправления
df['Сумма отправления'] = df['Итоговая стоимость'] * df['Количество товаров в отправлении']

# Добавляем финансовые показатели
df['Скидка, руб.'] = df['Итоговая стоимость'] * df['Скидка %'] / 100
df['Вознаграждение %'] = np.random.uniform(10, 20, n_records).astype(int)
df['Вознаграждение, руб.'] = df['Сумма отправления'] * df['Вознаграждение %'] / 100
df['Стоимость доставки'] = np.random.uniform(100, 500, n_records).astype(int)

# Рассчитываем к выплате
df['К выплате'] = df['Сумма отправления'] - df['Скидка, руб.'] - df['Вознаграждение, руб.'] - df['Стоимость доставки']

# Добавляем аналитические данные
df['Склад отгрузки'] = np.random.choice(['Склад 1', 'Склад 2', 'Склад 3'], n_records)

# Генерируем случайные даты в диапазоне 30 дней
start_date = datetime(2025, 1, 1)
df['Дата логистической операции'] = [start_date + timedelta(days=random.randint(0, 270)) for _ in range(n_records)]

# Сохраняем в CSV
df.to_csv('ozon_sales_data.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

# Выводим первые строки для проверки
print(df.head())


  Номер отправления Статус отправления   Ozon ID     Артикул товара  \
0         ORD479150            Отменен  25673692  HHH_Titanium_2410   
1         ORD290474          Доставлен  10991330     GGG_Green_7035   
2         ORD775156          Доставлен  96985825    HHH_Purple_2504   
3         ORD269078          Доставлен  41515025    DDD_Purple_3677   
4         ORD857346             В пути  60822434      GGG_Pink_4564   

   Итоговая стоимость  Количество товаров в отправлении  Скидка %  \
0              8500.0                                 1        15   
1              2600.0                                 4         0   
2              3200.0                                 1        15   
3               980.0                                 4         5   
4              9100.0                                 1         5   

                   Акции     Регион доставки Способ доставки  \
0                    Нет        с. Челябинск        Постамат   
1         Черная пятница      